In [ ]:
%run "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_01_Bronze/Jupyter Notebooks/00_bronze_data_connector.ipynb"

Preconnectors ran successfully and connected to 'clientdatastorage' storage accounts
 'policy_df, claim_df, member_df, provider_df' are ready to use from container 'raw data'(meridian architecture of bronze) 
 'policy_df, claim_df, member_df, provider_df' are ready to use from container 'raw data'(meridian architecture of bronze) 


In [40]:
import sys
import importlib

# Add the Silver layer directory to sys.path BEFORE importing
silver_dir = "/Users/manojrammopati/Public/Projects/bupa_insurance_project/02_Silver_Layer"
if silver_dir not in sys.path:
    sys.path.insert(0, silver_dir)

# Now import (and reload to pick up edits)
import utils_silver
from utils_silver import *

importlib.reload(utils_silver)

paths_bronze, paths_silver = utils_silver.build_paths(storage_account1)
print(sorted(paths_silver.keys()))
print("Imported utils_silver from", utils_silver.__file__)

✅ utils_silver.py loaded
['_metrics', '_quarantine', '_ref_dim_channel', '_ref_dim_product_line', '_reference', 'claims', 'members', 'policies', 'providers']
Imported utils_silver from /Users/manojrammopati/Public/Projects/bupa_insurance_project/02_Silver_Layer/utils_silver.py


In [ ]:
# Replace with your actual storage account and path
fact_policies = spark.read.format("delta").load("abfss://golddata@clientdatastorage.dfs.core.windows.net/fact_policies")
fact_members  = spark.read.format("delta").load("abfss://silverdata@clientdatastorage.dfs.core.windows.net/gold/fact_members")
fact_claims   = spark.read.format("delta").load("abfss://silverdata@clientdatastorage.dfs.core.windows.net/gold/fact_claims")

In [43]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS bupa_gold.fact_policies
    USING DELTA
    LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/fact_policies'
""")

DataFrame[]

In [44]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS bupa_gold.fact_members
    USING DELTA
    LOCATION 'abfss://silverdata@clientdatastorage.dfs.core.windows.net/gold/fact_members'
""")

DataFrame[]

In [45]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS bupa_gold.fact_claims
    USING DELTA
    LOCATION 'abfss://silverdata@clientdatastorage.dfs.core.windows.net/gold/fact_claims'
""")

DataFrame[]

# Cell 1 — Imports & load Gold fact tables

In [46]:
# Cell 1 — Setup: imports & config

from pyspark.sql import functions as F

DB_GOLD = "bupa_gold"

# Make sure database exists
spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_GOLD}")

# Load fact tables (already created in previous Gold notebooks)
fact_policies = spark.table(f"{DB_GOLD}.fact_policies")
fact_members  = spark.table(f"{DB_GOLD}.fact_members")
fact_claims   = spark.table(f"{DB_GOLD}.fact_claims")

print("fact_policies:", fact_policies.count())
print("fact_members :", fact_members.count())
print("fact_claims  :", fact_claims.count())

fact_policies.limit(5).show(truncate=False)


fact_policies: 381109
fact_members : 381109
fact_members : 381109
fact_claims  : 558211
fact_claims  : 558211


+----------+------------+------------+-------+------------------+------------------+-----------------+---------------+--------------------+--------------------+---------------------+------------------+-----------+-----------------+-------------+---------------+--------------+-----------------+----------------+--------------------------+----------------------------+----------------------------------------------------------------+
|Policy_ID |Customer_ID |Product_Line|Channel|Sum_Insured_GBP   |Annual_Premium_GBP|Policy_Start_Date|Policy_End_Date|Policy_Duration_Days|Renewal_Offered_Flag|Renewal_Accepted_Flag|Renewal_Conversion|Tenure_Band|Premium_Band     |Discount_Band|Renewal_Outcome|dq_money_valid|dq_discount_valid|dq_renewal_valid|created_at                |source_system               |record_hash                                                     |
+----------+------------+------------+-------+------------------+------------------+-----------------+---------------+----------------

# Cell 2 — dim_channel (sales channel dimension)

In [47]:
# Cell 2 — Build dim_channel from fact_policies

dim_channel = (
    fact_policies
      .select(F.col("Channel").alias("Channel_Name"))
      .where(F.col("Channel_Name").isNotNull())
      .withColumn(
          "Channel_Code",
          F.upper(F.regexp_replace(F.trim("Channel_Name"), r"\s+", "_"))
      )
      .dropDuplicates(["Channel_Code"])
      .select("Channel_Code", "Channel_Name")
      .orderBy("Channel_Code")
)

dim_channel.show(truncate=False)

# Save as managed Delta table in bupa_gold
dim_channel.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{DB_GOLD}.dim_channel")

print("✅ dim_channel written:", f"{DB_GOLD}.dim_channel")


+------------+------------+
|Channel_Code|Channel_Name|
+------------+------------+
|AGENT       |Agent       |
|BROKER      |Broker      |
|ONLINE      |Online      |
|PARTNER     |Partner     |
+------------+------------+



✅ dim_channel written: bupa_gold.dim_channel


# Cell 3 — dim_product_line

In [48]:
# Cell 3 — Build dim_product_line from fact_policies

dim_product_line = (
    fact_policies
      .select(F.col("Product_Line"))
      .where(F.col("Product_Line").isNotNull())
      .withColumn("Product_Line_Code",
                  F.upper(F.regexp_replace(F.trim("Product_Line"), r"\s+", "_")))
      .dropDuplicates(["Product_Line_Code"])
      .select("Product_Line_Code", "Product_Line")
      .orderBy("Product_Line_Code")
)

dim_product_line.show(truncate=False)

dim_product_line.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{DB_GOLD}.dim_product_line")

print("✅ dim_product_line written:", f"{DB_GOLD}.dim_product_line")


+-----------------+------------+
|Product_Line_Code|Product_Line|
+-----------------+------------+
|ACCIDENT         |Accident    |
|DENTAL           |Dental      |
|HEALTH           |Health      |
|MOTOR            |Motor       |
|TRAVEL           |Travel      |
+-----------------+------------+



✅ dim_product_line written: bupa_gold.dim_product_line


# Cell 4 — dim_region (member region dimension)

In [49]:
# Cell 4 — Build dim_region from fact_members

dim_region = (
    fact_members
      .select(F.col("Region"))
      .where(F.col("Region").isNotNull())
      .withColumn("Region_Code", F.col("Region"))
      # Simple normalised label; you already have Region_Group in fact_members
      .withColumn(
          "Region_Label",
          F.concat(F.lit("Region_"), F.col("Region_Code"))
      )
      .dropDuplicates(["Region_Code"])
      .orderBy("Region_Code")
)

dim_region.show(10, truncate=False)

dim_region.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{DB_GOLD}.dim_region")

print("✅ dim_region written:", f"{DB_GOLD}.dim_region")


+------+-----------+------------+
|Region|Region_Code|Region_Label|
+------+-----------+------------+
|00    |00         |Region_00   |
|10    |10         |Region_10   |
|100   |100        |Region_100  |
|110   |110        |Region_110  |
|120   |120        |Region_120  |
|130   |130        |Region_130  |
|140   |140        |Region_140  |
|150   |150        |Region_150  |
|160   |160        |Region_160  |
|170   |170        |Region_170  |
+------+-----------+------------+
only showing top 10 rows



✅ dim_region written: bupa_gold.dim_region


# Cell 5 — dim_claim_type

In [50]:
# Cell 5 — Build dim_claim_type from fact_claims

dim_claim_type = (
    fact_claims
      .select(F.col("Claim_Type"))
      .where(F.col("Claim_Type").isNotNull())
      .withColumn("Claim_Type_Code",
                  F.upper(F.regexp_replace(F.trim("Claim_Type"), r"\s+", "_")))
      .dropDuplicates(["Claim_Type_Code"])
      .select("Claim_Type_Code", "Claim_Type")
      .orderBy("Claim_Type_Code")
)

dim_claim_type.show(truncate=False)

dim_claim_type.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{DB_GOLD}.dim_claim_type")

print("✅ dim_claim_type written:", f"{DB_GOLD}.dim_claim_type")


+---------------+----------+
|Claim_Type_Code|Claim_Type|
+---------------+----------+
|DENTAL         |Dental    |
|HOSPITAL       |Hospital  |
|MATERNITY      |Maternity |
|OUTPATIENT     |Outpatient|
|PHYSIO         |Physio    |
|TRAVEL         |Travel    |
+---------------+----------+



✅ dim_claim_type written: bupa_gold.dim_claim_type


# Cell 6 — dim_member_segment (optional “risk segment” dimension)

This uses the banded columns you already created in fact_members.

In [51]:
# Cell 6 — Build dim_member_segment from fact_members (optional but powerful)

segment_cols = [
    "Age_Band",
    "BMI_Band",
    "Smoker_Status",
    "Chronic_Flag",
    "Chronic_Group",
    "Employment_Group",
    "Region_Group",
]

# Keep only segmentation columns & drop exact duplicates
dim_member_segment = (
    fact_members
      .select(*segment_cols)
      .dropDuplicates()
      .withColumn(
          "Member_Segment_Key",
          F.monotonically_increasing_id()
      )
      .select("Member_Segment_Key", *segment_cols)
      .orderBy("Member_Segment_Key")
)

dim_member_segment.show(10, truncate=False)

dim_member_segment.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{DB_GOLD}.dim_member_segment")

print("✅ dim_member_segment written:", f"{DB_GOLD}.dim_member_segment")


+------------------+--------+----------+-------------+------------+-------------+----------------+--------------+
|Member_Segment_Key|Age_Band|BMI_Band  |Smoker_Status|Chronic_Flag|Chronic_Group|Employment_Group|Region_Group  |
+------------------+--------+----------+-------------+------------+-------------+----------------+--------------+
|0                 |18–29   |Obese     |Non-Smoker   |1           |Hypertension |Employed        |Region_150-249|
|1                 |18–29   |Overweight|Smoker       |0           |None         |Employed        |Region_350+   |
|2                 |45–59   |Obese     |Non-Smoker   |1           |Diabetes     |Retired         |Region_0-49   |
|3                 |18–29   |Obese     |Smoker       |1           |Asthma       |Retired         |Region_50-149 |
|4                 |45–59   |Overweight|Smoker       |1           |Hypertension |Student         |Region_50-149 |
|5                 |30–44   |Obese     |Non-Smoker   |0           |None         |Retired

✅ dim_member_segment written: bupa_gold.dim_member_segment


# Cell 7 — Quick sanity checks (row counts)

In [52]:
# Cell 7 — Sanity checks: dimension sizes

print("dim_channel        :", spark.table(f"{DB_GOLD}.dim_channel").count())
print("dim_product_line   :", spark.table(f"{DB_GOLD}.dim_product_line").count())
print("dim_region         :", spark.table(f"{DB_GOLD}.dim_region").count())
print("dim_claim_type     :", spark.table(f"{DB_GOLD}.dim_claim_type").count())
print("dim_member_segment :", spark.table(f"{DB_GOLD}.dim_member_segment").count())


dim_channel        : 4
dim_product_line   : 5
dim_product_line   : 5
dim_region         : 53
dim_region         : 53
dim_claim_type     : 6
dim_claim_type     : 6
dim_member_segment : 1440
dim_member_segment : 1440


# Cell 8 — Join test: fact_policies ↔ dim_channel

In [53]:
# Cell 8 — Join test: fact_policies with dim_channel

dim_channel = spark.table(f"{DB_GOLD}.dim_channel")

test_join = (
    fact_policies.alias("f")
      .join(dim_channel.alias("d"),
            F.upper(F.regexp_replace(F.col("f.Channel"), r"\s+", "_")) == F.col("d.Channel_Code"),
            "left"
      )
)

missing_channels = test_join.filter(F.col("d.Channel_Code").isNull()).count()
total_rows = test_join.count()

print(f"Total policy rows: {total_rows}")
print(f"Rows without matching dim_channel: {missing_channels}")


Total policy rows: 381109
Rows without matching dim_channel: 0


# Cell 9 — Join test: fact_claims ↔ dim_claim_type

In [54]:
# Cell 9 — Join test: fact_claims with dim_claim_type

dim_claim_type = spark.table(f"{DB_GOLD}.dim_claim_type")

test_join_ct = (
    fact_claims.alias("f")
      .join(dim_claim_type.alias("d"),
            F.upper(F.regexp_replace(F.col("f.Claim_Type"), r'\s+', "_")) == F.col("d.Claim_Type_Code"),
            "left"
      )
)

missing_ct = test_join_ct.filter(F.col("d.Claim_Type_Code").isNull()).count()
total_claims = test_join_ct.count()

print(f"Total claim rows: {total_claims}")
print(f"Rows without matching dim_claim_type: {missing_ct}")


Total claim rows: 558211
Rows without matching dim_claim_type: 39196


# Cell 10 — Final summary

In [55]:
# Cell 10 — Final summary print

print("✅ Gold Dimension Tables created in database:", DB_GOLD)
print("   - dim_channel")
print("   - dim_product_line")
print("   - dim_region")
print("   - dim_claim_type")
print("   - dim_member_segment (segmentation dim)")

for tbl in ["dim_channel", "dim_product_line", "dim_region",
            "dim_claim_type", "dim_member_segment"]:
    print(f"\n{tbl}:")
    spark.table(f"{DB_GOLD}.{tbl}").show(5, truncate=False)


✅ Gold Dimension Tables created in database: bupa_gold
   - dim_channel
   - dim_product_line
   - dim_region
   - dim_claim_type
   - dim_member_segment (segmentation dim)

dim_channel:
+------------+------------+
|Channel_Code|Channel_Name|
+------------+------------+
|AGENT       |Agent       |
|BROKER      |Broker      |
|ONLINE      |Online      |
|PARTNER     |Partner     |
+------------+------------+


dim_product_line:
+------------+------------+
|Channel_Code|Channel_Name|
+------------+------------+
|AGENT       |Agent       |
|BROKER      |Broker      |
|ONLINE      |Online      |
|PARTNER     |Partner     |
+------------+------------+


dim_product_line:
+-----------------+------------+
|Product_Line_Code|Product_Line|
+-----------------+------------+
|ACCIDENT         |Accident    |
|DENTAL           |Dental      |
|HEALTH           |Health      |
|MOTOR            |Motor       |
|TRAVEL           |Travel      |
+-----------------+------------+


dim_region:
+-------------

# 1) Define Gold paths (do this once, e.g. in Cell 1)

In [56]:
storage_account = "clientdatastorage"
GOLD_CONTAINER = "golddata"
GOLD_BASE = f"abfss://{GOLD_CONTAINER}@{storage_account}.dfs.core.windows.net"

paths_gold_dim = {
    "dim_channel":       f"{GOLD_BASE}/dim_channel",
    "dim_product_line":  f"{GOLD_BASE}/dim_product_line",
    "dim_region":        f"{GOLD_BASE}/dim_region",
    "dim_claim_type":    f"{GOLD_BASE}/dim_claim_type",
    "dim_member_segment":f"{GOLD_BASE}/dim_member_segment",
}


# 2) Write dim_channel to ADLS + register table

In [57]:
# Write and register all Gold dimension tables with explicit path variables
dim_channel_path        = paths_gold_dim["dim_channel"]
dim_product_line_path  = paths_gold_dim["dim_product_line"]
dim_region_path        = paths_gold_dim["dim_region"]
dim_claim_type_path    = paths_gold_dim["dim_claim_type"]
dim_member_segment_path= paths_gold_dim["dim_member_segment"]

# dim_channel
(dim_channel.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(dim_channel_path))
print("✅ dim_channel written to:", dim_channel_path)
spark.sql(f"DROP TABLE IF EXISTS {DB_GOLD}.dim_channel")
spark.sql(f"""
CREATE TABLE {DB_GOLD}.dim_channel
USING DELTA
LOCATION '{dim_channel_path}'
""")
print("✅ dim_channel registered as:", f"{DB_GOLD}.dim_channel")

# dim_product_line
(dim_product_line.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(dim_product_line_path))
print("✅ dim_product_line written to:", dim_product_line_path)
spark.sql(f"DROP TABLE IF EXISTS {DB_GOLD}.dim_product_line")
spark.sql(f"""
CREATE TABLE {DB_GOLD}.dim_product_line
USING DELTA
LOCATION '{dim_product_line_path}'
""")
print("✅ dim_product_line registered as:", f"{DB_GOLD}.dim_product_line")

# dim_region
(dim_region.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(dim_region_path))
print("✅ dim_region written to:", dim_region_path)
spark.sql(f"DROP TABLE IF EXISTS {DB_GOLD}.dim_region")
spark.sql(f"""
CREATE TABLE {DB_GOLD}.dim_region
USING DELTA
LOCATION '{dim_region_path}'
""")
print("✅ dim_region registered as:", f"{DB_GOLD}.dim_region")

# dim_claim_type
(dim_claim_type.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(dim_claim_type_path))
print("✅ dim_claim_type written to:", dim_claim_type_path)
spark.sql(f"DROP TABLE IF EXISTS {DB_GOLD}.dim_claim_type")
spark.sql(f"""
CREATE TABLE {DB_GOLD}.dim_claim_type
USING DELTA
LOCATION '{dim_claim_type_path}'
""")
print("✅ dim_claim_type registered as:", f"{DB_GOLD}.dim_claim_type")

# dim_member_segment
(dim_member_segment.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(dim_member_segment_path))
print("✅ dim_member_segment written to:", dim_member_segment_path)
spark.sql(f"DROP TABLE IF EXISTS {DB_GOLD}.dim_member_segment")
spark.sql(f"""
CREATE TABLE {DB_GOLD}.dim_member_segment
USING DELTA
LOCATION '{dim_member_segment_path}'
""")
print("✅ dim_member_segment registered as:", f"{DB_GOLD}.dim_member_segment")

✅ dim_channel written to: abfss://golddata@clientdatastorage.dfs.core.windows.net/dim_channel
✅ dim_channel registered as: bupa_gold.dim_channel


✅ dim_product_line written to: abfss://golddata@clientdatastorage.dfs.core.windows.net/dim_product_line
✅ dim_product_line registered as: bupa_gold.dim_product_line


✅ dim_region written to: abfss://golddata@clientdatastorage.dfs.core.windows.net/dim_region
✅ dim_region registered as: bupa_gold.dim_region


✅ dim_claim_type written to: abfss://golddata@clientdatastorage.dfs.core.windows.net/dim_claim_type
✅ dim_claim_type registered as: bupa_gold.dim_claim_type


✅ dim_member_segment written to: abfss://golddata@clientdatastorage.dfs.core.windows.net/dim_member_segment
✅ dim_member_segment registered as: bupa_gold.dim_member_segment
